# 01 · The baseline, measured

No training yet. A candidate scores by how close it sits to what I've already logged, and this checks whether that score finds the newest fifth of my log when it's hidden.

It loads my items, the catalogs and every set of embeddings, splits the log by time, ranks the held-out items among each medium's catalog, scores that per model, and then looks at what it found and missed. `uv run evaluate` prints the same table from `src/recs/evaluate.py` without the rest.

In [ ]:
import json
import numpy as np
import pandas as pd

from recs.config import default_model_key
from recs.evaluate import KS, available_sets, is_complete, ranks_for, summarise, temporal_split
from recs.paths import FILE_NAMES, ITEMS, catalog_path, embedding_dir, embedding_path
from recs.recommend import score_candidates

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)

items = pd.read_parquet(ITEMS)
catalogs = {m: pd.read_parquet(catalog_path(m)) for m in ("book", "movie", "show")}

sets = available_sets()           # every embeddings/<model>/ folder that has items.npy
main = default_model_key() if default_model_key() in sets else sets[0]
print("embedding sets:", sets, "— looking closely at:", main)
for key in sets:
    meta = json.loads((embedding_dir(key) / "meta.json").read_text())
    print(f"  {key}: {meta['model']} @ {meta['revision']}, {meta['dimensions']} dims, {meta['seconds']}s")

items["medium"].value_counts()

## Split by time

A random 80/20 split would be wrong here. Recommending is guessing what I'll log next from what I've logged so far, and with a random split the history would include things logged after the items it's tested on. That's leakage: the history knows the future.

So the oldest 80% is the history and the newest 20% is held out. Everything before 2019 was imported in bulk that year and only its log year is real, so the sort is on `(year_logged, date_logged)`.

The 20% is of everything, not per medium, so whatever I logged more of lately gets tested more.

In [ ]:
split = temporal_split(items)
print(f"history: {split.train.sum()}  held out: {split.heldout.sum()}  (cut at {split.cut_date})")
pd.crosstab(items["medium"], np.where(split.heldout, "held out", "history"))

## Rank the held-out items

Per medium, the history is that medium's items from before the cut, and the candidates are the catalog plus the held-out items. The catalog leaves out everything I've logged, so without adding them the baseline couldn't find the held-out items however good it was.

A candidate's score is its mean cosine similarity to its 5 nearest history items (`score_candidates` says why the top 5 and not all of them). Sort by score and see where each held-out item landed.

In [ ]:
vectors = {key: np.load(embedding_path(key, "items")) for key in sets}

ranks = {
    (key, medium): ranks_for(items, vectors[key], split, key, medium, k=5)
    for key in sets
    for medium in ("book", "movie", "show")
}
{k: (len(v), int(np.median(v))) for k, v in ranks.items()}  # (held out, median rank)

## recall@K, MRR, median rank

recall@K is the share of held-out items in the top K. Had I been shown the top 50 at the cut date, it's how many of the things I went on to log would have been on that list.

MRR, mean reciprocal rank, is the mean of 1/rank. It mostly rewards a few very high: one hit at rank 1 counts as much as ten at rank 10.

Median rank is where a typical held-out item lands. `chance` is what a random order would give, half the candidates, to show how far from random it is.

A row per medium and model, so the models can be read against each other.

In [ ]:
rows = []
for (key, medium), r in ranks.items():
    n_candidates = len(catalogs[medium]) + len(r)
    rows.append({"medium": medium, "model": key, "held_out": len(r), **summarise(r, n_candidates)})
table = pd.DataFrame(rows).set_index(["medium", "model"]).sort_index()
table.round(3)

The same, for held-out items that have a synopsis and genres. See the caveats at the end for why.

In [ ]:
rows = []
for (key, medium), r in ranks.items():
    held = items[split.heldout & (items["medium"] == medium)]
    r = r[is_complete(held)]
    n_candidates = len(catalogs[medium]) + len(held)
    rows.append({"medium": medium, "model": key, "held_out": len(r), **summarise(r, n_candidates)})
pd.DataFrame(rows).set_index(["medium", "model"]).sort_index().round(3)

## Hits and misses

For the main model, the held-out items that ranked highest, each with the two history items that pulled it up, which is all the explanation the baseline has. Then the ones it ranked lowest: things I logged that look nothing like what came before, or that the catalog has nothing near.

In [ ]:
def explain(key: str, medium: str, worst: bool = False, n: int = 8) -> pd.DataFrame:
    of_medium = (items["medium"] == medium).to_numpy()
    history = items[split.train & of_medium].reset_index(drop=True)
    heldout = items[split.heldout & of_medium].reset_index(drop=True)
    hv = vectors[key][split.train & of_medium]
    ov = vectors[key][split.heldout & of_medium]
    scores, nearest = score_candidates(ov, hv, k=5)
    r = ranks[(key, medium)]
    order = np.argsort(-r if worst else r)[:n]
    return pd.DataFrame({
        "rank": r[order],
        "title": heldout.loc[order, "title"].values,
        "year": heldout.loc[order, "year"].values,
        "because": [", ".join(history.loc[j, "title"] for j in nearest[i][:2]) for i in order],
    })

for medium in ("book", "movie", "show"):
    print(f"\n== {medium}: best found ==")
    display(explain(main, medium))
    print(f"== {medium}: missed ==")
    display(explain(main, medium, worst=True))

## Caveats

The catalog shapes the result. Films and shows come from TMDB discover, popular titles by genre and decade, and books from Open Library subjects picked from my whole log, held-out items included. That leaks a little in the books' favour. recall@K against a 2,000-item catalog also isn't recall against everything ever published.

Half of the held-out books have no synopsis or subjects, because Open Library has little on recent books, while the catalog books nearly all do. A text that is only a title and a year ranks on the year, which is recency, not taste. The second table above leaves those out, and that's the one to compare models on; the fix is a second source of descriptions for books.

One cut date is one sample. `uv run evaluate --rolling` averages over five, and `--half-life` weights recent history more, which turns out to matter: my taste in films moved, and a history where 2010 counts as much as last year keeps recommending 2010.

Nothing is trained, so nothing can overfit yet. Notebook 02 changes that, and this table is what it has to beat on the same split and candidates.

Re-reads and later seasons are collapsed to the first time I logged a work, so a re-read after the cut doesn't count as a hit. That would be too easy.